In [ ]:
# On va tester d'abord la connexion
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()


DB_HOST = os.getenv("DB_HOST")        
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "kayak")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD")


url = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(url)


try:
    with engine.connect() as conn:
        version = conn.execute(text("SELECT version();")).scalar()
        base = conn.execute(text("SELECT current_database();")).scalar()
    print("Connexion réussie")
    print("   Base :", base)
    print("   PostgreSQL :", version[:40])
except Exception as e:
    print(" Connexion échouée")
    print("  ", type(e).__name__, ":", str(e)[:200])



Connexion réussie
   Base : kayak
   PostgreSQL : PostgreSQL 18.3 on aarch64-unknown-linux


In [ ]:
# On recupère les fichiers depuis S3

import io
import boto3
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

s3 = boto3.client("s3", region_name="eu-west-3")

BUCKET = "aina-group-bucket"
BASE_PREFIX = "temp/tp-data-eng/jedha_certification_kayak"
CLE_ENRICHED = f"{BASE_PREFIX}/curated/kayak_enriched.csv"

# --- EXTRACT : lire le CSV directement depuis S3 ---
reponse = s3.get_object(Bucket=BUCKET, Key=CLE_ENRICHED)
contenu = reponse["Body"].read()                  
df = pd.read_csv(io.BytesIO(contenu))             

print("Extrait de S3 :", len(df), "lignes")
print("   Source : s3://" + BUCKET + "/" + CLE_ENRICHED)
df.head(3)

Extrait de S3 : 875 lignes
   Source : s3://aina-group-bucket/temp/tp-data-eng/jedha_certification_kayak/curated/kayak_enriched.csv


,hotel_nom,hotel_url,hotel_note,hotel_prix,hotel_lat,hotel_lon,city_id,city,ville_lat,ville_lon,temp_moy,clouds_moy,pop_moy,temp_score,ciel_score,pluie_score,score
0,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html,8.1,NaN,48.614700,-1.509617,1,Mont Saint Michel,48.635954,-1.51146,23.14125,91.375,0.9375,0.162751,0.061475,0.0,0.074742
1,Le Relais Saint Michel,https://www.booking.com/hotel/fr/le-relais-sai...,8.1,NaN,48.617587,-1.510396,1,Mont Saint Michel,48.635954,-1.51146,23.14125,91.375,0.9375,0.162751,0.061475,0.0,0.074742
2,Le Saint Aubert,https://www.booking.com/hotel/fr/hotel-saint-a...,7.4,NaN,48.612938,-1.510105,1,Mont Saint Michel,48.635954,-1.51146,23.14125,91.375,0.9375,0.162751,0.061475,0.0,0.074742


In [ ]:
# créons les tables

from sqlalchemy import text

schema_sql = """
DROP TABLE IF EXISTS hotels;
DROP TABLE IF EXISTS cities;

CREATE TABLE cities (
    city_id     INTEGER PRIMARY KEY,
    city        VARCHAR(100),
    ville_lat   DOUBLE PRECISION,
    ville_lon   DOUBLE PRECISION,
    temp_moy    DOUBLE PRECISION,
    clouds_moy  DOUBLE PRECISION,
    pop_moy     DOUBLE PRECISION,
    score       DOUBLE PRECISION
);

CREATE TABLE hotels (
    hotel_id    INTEGER PRIMARY KEY,
    city_id     INTEGER REFERENCES cities(city_id),
    hotel_nom   VARCHAR(300),
    hotel_url   TEXT,
    hotel_note  DOUBLE PRECISION,
    hotel_prix  DOUBLE PRECISION,
    hotel_lat   DOUBLE PRECISION,
    hotel_lon   DOUBLE PRECISION
);
"""

with engine.begin() as conn:          
    for statement in schema_sql.strip().split(";"):
        if statement.strip():
            conn.execute(text(statement))

print("Tables cities et hotels créées")

Tables cities et hotels créées


In [ ]:
# --- Table CITIES ---
colonnes_ville = ["city_id", "city", "ville_lat", "ville_lon",
                  "temp_moy", "clouds_moy", "pop_moy", "score"]
df_cities = df[colonnes_ville].drop_duplicates(subset="city_id").reset_index(drop=True)

# --- Table HOTELS ---
colonnes_hotel = ["city_id", "hotel_nom", "hotel_url", "hotel_note",
                  "hotel_prix", "hotel_lat", "hotel_lon"]
df_hotels_db = df[colonnes_hotel].reset_index(drop=True)

df_hotels_db.insert(0, "hotel_id", range(1, len(df_hotels_db) + 1))

print("cities :", len(df_cities), "lignes")   # ~35
print("hotels :", len(df_hotels_db), "lignes")  # ~873
df_cities.head(3)

cities : 35 lignes
hotels : 875 lignes


,city_id,city,ville_lat,ville_lon,temp_moy,clouds_moy,pop_moy,score
0,1,Mont Saint Michel,48.635954,-1.511460,23.14125,91.375,0.93750,0.074742
1,10,Chateau du Haut Koenigsbourg,48.249382,7.343941,23.36875,69.750,0.52375,0.351927
2,11,Colmar,48.077752,7.357964,25.84000,64.500,0.53375,0.462491


In [ ]:



df_cities.to_sql("cities", engine, if_exists="append", index=False)
print("cities chargée")

df_hotels_db.to_sql("hotels", engine, if_exists="append", index=False)
print("hotels chargée")

# --- Vérification ---
from sqlalchemy import text
with engine.connect() as conn:
    n_cities = conn.execute(text("SELECT COUNT(*) FROM cities")).scalar()
    n_hotels = conn.execute(text("SELECT COUNT(*) FROM hotels")).scalar()
print(f"\nEn base : {n_cities} villes, {n_hotels} hôtels")

cities chargée
hotels chargée

En base : 35 villes, 875 hôtels


In [7]:
#Testons la valeur

requete = """
SELECT c.city, c.score, COUNT(h.hotel_id) AS nb_hotels, ROUND(AVG(h.hotel_note)::numeric, 2) AS note_moy
FROM cities c
JOIN hotels h ON c.city_id = h.city_id
GROUP BY c.city, c.score
ORDER BY c.score DESC
LIMIT 10;
"""
pd.read_sql(requete, engine)

,city,score,nb_hotels,note_moy
0,Collioure,0.851248,25,8.83
1,Aix en Provence,0.830380,25,8.71
2,Avignon,0.804169,25,7.95
3,Carcassonne,0.792824,25,8.90
4,Grenoble,0.790080,25,8.26
5,Marseille,0.759568,25,7.64
6,Nimes,0.756464,25,8.38
7,Cassis,0.733013,25,8.91
8,Uzes,0.721901,25,8.93
9,Bormes les Mimosas,0.686372,25,8.60


In [8]:
# On recupère le top_20 des hotels

import pandas as pd

requete = """
WITH top_villes AS (
    SELECT city_id, city, score
    FROM cities
    ORDER BY score DESC
    LIMIT 5
)
SELECT h.hotel_nom, h.hotel_note, h.hotel_lat, h.hotel_lon,
       h.hotel_url, v.city, v.score AS score_meteo
FROM hotels h
JOIN top_villes v ON h.city_id = v.city_id
WHERE h.hotel_note IS NOT NULL
  AND h.hotel_lat IS NOT NULL
ORDER BY h.hotel_note DESC
LIMIT 20;
"""

top20 = pd.read_sql(requete, engine)
print(len(top20), "hôtels")
top20

20 hôtels


,hotel_nom,hotel_note,hotel_lat,hotel_lon,hotel_url,city,score_meteo
0,Situation exceptionnelle HARMONY III,9.9,43.211814,2.352205,https://www.booking.com/hotel/fr/harmony-3.fr....,Carcassonne,0.792824
1,La Parenthèse Saint Donat,9.8,43.553075,5.454037,https://www.booking.com/hotel/fr/la-parenthese...,Aix en Provence,0.830380
2,Sur le quai,9.8,43.209339,2.355713,https://www.booking.com/hotel/fr/sur-le-quai-c...,Carcassonne,0.792824
3,Le Belvédère St Gimer,9.8,43.206833,2.361293,https://www.booking.com/hotel/fr/le-belvedere-...,Carcassonne,0.792824
4,"Villa Montplaisir, maison avec piscine privée ...",9.7,43.950111,4.829684,https://www.booking.com/hotel/fr/villa-montpla...,Avignon,0.804169
5,cocon bohéme,9.7,43.530957,5.446107,https://www.booking.com/hotel/fr/cocon-boheme-...,Aix en Provence,0.830380
6,La Farigoule - Sainte-Victoire - Vue exception...,9.7,43.531148,5.474076,https://www.booking.com/hotel/fr/la-farigoule-...,Aix en Provence,0.830380
7,La Villa Rustica,9.7,43.558793,5.448861,https://www.booking.com/hotel/fr/la-villa-rust...,Aix en Provence,0.830380
8,45BB,9.7,43.210256,2.350132,https://www.booking.com/hotel/fr/45bb.fr.html,Carcassonne,0.792824
9,SalutBB Chambre d'hote,9.7,43.211449,2.346299,https://www.booking.com/hotel/fr/salut-chambre...,Carcassonne,0.792824


In [ ]:

tableau = top20[["hotel_nom", "city", "hotel_note"]].copy()


tableau = tableau.rename(columns={
    "hotel_nom": "Hôtel",
    "city": "Ville",
    "hotel_note": "Note",
})

# On classe par note décroissante et on remet un numéro de rang propre
tableau = tableau.sort_values("Note", ascending=False).reset_index(drop=True)
tableau.index = tableau.index + 1        

tableau

,Hôtel,Ville,Note
1,Situation exceptionnelle HARMONY III,Carcassonne,9.9
2,La Parenthèse Saint Donat,Aix en Provence,9.8
3,Sur le quai,Carcassonne,9.8
4,Le Belvédère St Gimer,Carcassonne,9.8
5,"Villa Montplaisir, maison avec piscine privée ...",Avignon,9.7
6,cocon bohéme,Aix en Provence,9.7
7,La Farigoule - Sainte-Victoire - Vue exception...,Aix en Provence,9.7
8,La Villa Rustica,Aix en Provence,9.7
9,45BB,Carcassonne,9.7
10,SalutBB Chambre d'hote,Carcassonne,9.7


In [9]:
# Affichons l'hotel des top regions
import plotly.express as px

fig = px.scatter_map(
    top20,
    lat="hotel_lat",
    lon="hotel_lon",
    text="hotel_nom",
    color="hotel_note",
    size="hotel_note",
    size_max=22,
    color_continuous_scale="RdYlGn",
    hover_name="hotel_nom",
    hover_data={"city": True, "hotel_note": ":.1f",
                "hotel_lat": False, "hotel_lon": False},
    zoom=5,
    height=650,
)
fig.update_traces(textposition="top center")
fig.update_layout(
    map_style="carto-positron",
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    title="Top 20 hôtels des meilleures destinations",
)
fig.show()